In [ ]:
import os
import cv2
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms

from tqdm import tqdm

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
CKPT_PATH = "/content/drive/MyDrive/dino_finetuning_outputs/ckpt_dino_finetuning_best.pth"
INPUT_VIDEO = "/content/drive/MyDrive/dashcam_input.mp4"
OUTPUT_VIDEO = "/content/dashcam_demo_sidebyside.mp4"

INFER_HW = (1024, 2048)
PANE_HW = (512, 1024)
BATCH_SIZE = 16
NUM_CLASSES = 19
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
class DinoBackbone(nn.Module):
    def __init__(self, vit, patch_size=16, n_main=4, return_aux=True):
        super().__init__()
        self.vit = vit
        self.patch_size = patch_size
        self.n_main = n_main
        self.return_aux = return_aux
        self.embed_dim = vit.embed_dim  # 384 for ViT-S

    def forward(self, x):
        B, _, H, W = x.shape
        n = self.n_main + (1 if self.return_aux else 0)
        feats = self.vit.get_intermediate_layers(x, n=n)
        h_p = H // self.patch_size
        w_p = W // self.patch_size
        spatial = []
        for f in feats:
            f = f[:, 1:, :].reshape(B, h_p, w_p, -1).permute(0, 3, 1, 2).contiguous()
            spatial.append(f)
        if self.return_aux:
            return spatial[1:], spatial[0]
        return spatial, None


def build_dino_backbone(n_main=4, return_aux=True):
    print("  Loading DINO ViT-S/16 from torch.hub...")
    vit = torch.hub.load("facebookresearch/dino:main", "dino_vits16", pretrained=True)
    return DinoBackbone(vit, patch_size=16, n_main=n_main, return_aux=return_aux)

In [ ]:
class DPTLiteHead(nn.Module):
    def __init__(self, in_channels=384, n_layers=4, num_classes=19, proj_channels=128):
        super().__init__()
        self.proj = nn.ModuleList([nn.Conv2d(in_channels, proj_channels, kernel_size=1) for _ in range(n_layers)])
        fused = proj_channels * n_layers
        self.fusion = nn.Sequential(
            nn.Conv2d(fused, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )
        self.up1 = self._up_block(256, 128)
        self.up2 = self._up_block(128,  64)
        self.up3 = self._up_block( 64,  32)
        self.up4 = self._up_block( 32,  32)
        self.classifier = nn.Conv2d(32, num_classes, kernel_size=1)

    @staticmethod
    def _up_block(in_ch, out_ch):
        return nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, feats):
        x = torch.cat([p(f) for p, f in zip(self.proj, feats)], dim=1)
        x = self.fusion(x)
        x = self.up1(x); x = self.up2(x); x = self.up3(x); x = self.up4(x)
        return self.classifier(x)


class AuxHead(nn.Module):
    def __init__(self, in_channels=384, num_classes=19, hidden=128, scale=16):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, hidden, kernel_size=3, padding=1),
            nn.BatchNorm2d(hidden),
            nn.ReLU(inplace=True),
        )
        self.classifier = nn.Conv2d(hidden, num_classes, kernel_size=1)
        self.scale = scale

    def forward(self, x):
        x = self.conv(x)
        x = self.classifier(x)
        return F.interpolate(x, scale_factor=self.scale, mode="bilinear", align_corners=False)


class SegmentationModel(nn.Module):
    def __init__(self, backbone, head, aux_head=None):
        super().__init__()
        self.backbone = backbone
        self.head = head
        self.aux_head = aux_head

    def forward(self, x):
        main_feats, aux_feat = self.backbone(x)
        main_out = self.head(main_feats)
        if self.training and self.aux_head is not None and aux_feat is not None:
            aux_out = self.aux_head(aux_feat)
            return main_out, aux_out
        return main_out


def build_model():
    backbone = build_dino_backbone(n_main=4, return_aux=True)
    head = DPTLiteHead(in_channels=backbone.embed_dim, n_layers=4, num_classes=NUM_CLASSES)
    aux  = AuxHead(in_channels=backbone.embed_dim, num_classes=NUM_CLASSES, scale=16)
    return SegmentationModel(backbone, head, aux).to(DEVICE)

In [ ]:
class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {}
        for n, p in model.named_parameters():
            if p.dtype.is_floating_point:
                self.shadow[n] = p.detach().clone()
        for n, b in model.named_buffers():
            if b.dtype.is_floating_point:
                self.shadow[n] = b.detach().clone()

    @torch.no_grad()
    def update(self, model):
        d = self.decay
        for n, p in model.named_parameters():
            if n in self.shadow:
                self.shadow[n].mul_(d).add_(p.detach(), alpha=1.0 - d)
        for n, b in model.named_buffers():
            if n in self.shadow:
                self.shadow[n].copy_(b.detach())

    @torch.no_grad()
    def with_ema(self, model):
        backup = {}
        for n, p in model.named_parameters():
            if n in self.shadow:
                backup[n] = p.detach().clone()
                p.data.copy_(self.shadow[n])
        for n, b in model.named_buffers():
            if n in self.shadow:
                backup[n] = b.detach().clone()
                b.data.copy_(self.shadow[n])
        return backup

    @torch.no_grad()
    def restore(self, model, backup):
        for n, p in model.named_parameters():
            if n in backup:
                p.data.copy_(backup[n])
        for n, b in model.named_buffers():
            if n in backup:
                b.data.copy_(backup[n])

    def state_dict(self):
        return {k: v.detach().clone() for k, v in self.shadow.items()}

    def load_state_dict(self, state):
        for k, v in state.items():
            if k in self.shadow:
                self.shadow[k].copy_(v)

In [ ]:
CLASS_NAMES = [
    "road", "sidewalk", "building", "wall", "fence", "pole",
    "traffic light", "traffic sign", "vegetation", "terrain", "sky",
    "person", "rider", "car", "truck", "bus", "train",
    "motorcycle", "bicycle",
]

CLASS_COLORS = np.array([
    [128,  64, 128], [244,  35, 232], [ 70,  70,  70], [102, 102, 156],
    [190, 153, 153], [153, 153, 153], [250, 170,  30], [220, 220,   0],
    [107, 142,  35], [152, 251, 152], [ 70, 130, 180], [220,  20,  60],
    [255,   0,   0], [  0,   0, 142], [  0,   0,  70], [  0,  60, 100],
    [  0,  80, 100], [  0,   0, 230], [119,  11,  32],
], dtype=np.uint8)

def colorize_mask(mask):
    out = np.zeros((*mask.shape, 3), dtype=np.uint8)
    for cls in range(NUM_CLASSES):
        out[mask == cls] = CLASS_COLORS[cls]
    return out

In [ ]:
model = build_model()
ema = EMA(model, decay=0.999)

ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["model"])
ema.load_state_dict(ckpt["ema"])
ema.with_ema(model)
model.eval()

print(f"Loaded checkpoint: epoch={ckpt['epoch']}, val mIoU={ckpt['miou']:.4f}")

In [ ]:
NORMALIZE = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

def preprocess_frame(bgr):
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    H_t, W_t = INFER_HW
    h, w = rgb.shape[:2]
    new_h = int(round(h * W_t / w))
    rgb = cv2.resize(rgb, (W_t, new_h), interpolation=cv2.INTER_LINEAR)
    if new_h >= H_t:
        top = (new_h - H_t) // 2
        rgb = rgb[top:top + H_t]
    else:
        pad = H_t - new_h
        rgb = np.pad(rgb, ((pad // 2, pad - pad // 2), (0, 0), (0, 0)))
    img_t = torch.from_numpy(rgb).permute(2, 0, 1).float() / 255.0
    img_t = NORMALIZE(img_t)
    return img_t, rgb

In [ ]:
cap = cv2.VideoCapture(INPUT_VIDEO)
assert cap.isOpened(), f"Could not open {INPUT_VIDEO}"
fps = cap.get(cv2.CAP_PROP_FPS)
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"Input: {n_frames} frames @ {fps:.2f} fps")

pane_h, pane_w = PANE_HW
out_w, out_h = pane_w * 2, pane_h
writer = cv2.VideoWriter(OUTPUT_VIDEO, cv2.VideoWriter_fourcc(*"mp4v"), fps, (out_w, out_h))

@torch.no_grad()
def run_batch(tensors, raws):
    x = torch.stack(tensors).to(DEVICE, non_blocking=True)
    with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        logits = model(x)
    preds = logits.argmax(dim=1).cpu().numpy().astype(np.uint8)
    for raw, pred in zip(raws, preds):
        seg = colorize_mask(pred)
        left = cv2.resize(raw, (pane_w, pane_h), interpolation=cv2.INTER_AREA)
        right = cv2.resize(seg, (pane_w, pane_h), interpolation=cv2.INTER_NEAREST)
        combined = np.concatenate([left, right], axis=1)
        writer.write(cv2.cvtColor(combined, cv2.COLOR_RGB2BGR))

batch_t, batch_raw = [], []
pbar = tqdm(total=n_frames)
while True:
    ok, frame = cap.read()
    if not ok:
        break
    t, raw = preprocess_frame(frame)
    batch_t.append(t); batch_raw.append(raw)
    if len(batch_t) == BATCH_SIZE:
        run_batch(batch_t, batch_raw)
        batch_t, batch_raw = [], []
    pbar.update(1)
if batch_t:
    run_batch(batch_t, batch_raw)
pbar.close()
cap.release()
writer.release()
print(f"Wrote {OUTPUT_VIDEO}")